# Section 4 – Validation Protocol and Metrics

This notebook covers the **backtesting (walk-forward validation) framework** used
throughout the lab to evaluate energy-consumption forecasting models.

Topics
------
1. The `backtest()` function — parameters and return value  
2. Baseline model: Seasonal Naïve  
3. Error metrics: MAE, RMSE, WAPE, MAPE, MASE  
4. MAE by forecast horizon (h = 1 … 24)  
5. MAE by hour of day  
6. Christmas week error analysis (Dec 22–28)  
7. Walk-forward validation with **monthly retraining**  
8. Why MAPE fails with near-zero consumption  


---
## 0. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (11, 4)})
print("Setup complete.")


---
## 1. The `backtest()` Function

`src/backtest.py` contains the main entry-point for evaluating any forecasting model
on an energy dataset.  Let's look at its signature and understand every parameter.


In [ ]:
# Import the function from src/backtest.py
from backtest import backtest
help(backtest)


### Parameter guide

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `model` | object | — | Any object with `.fit(X, y)` and `.predict(X)` |
| `dataset` | `pd.DataFrame` | — | DatetimeIndex + column `"y"` (target) + optional features |
| `config` | dict | `{}` | Dictionary of behavioural knobs (see below) |

**`config` keys**

| Key | Default | Meaning |
|-----|---------|--------|
| `horizon` | 24 | Number of future steps to forecast each window |
| `initial_train` | 8760 | Minimum observations before first forecast (1 year hourly) |
| `step` | 24 | How many observations to advance the window between iterations |
| `monthly_retrain` | `True` | Re-fit the model at the start of each new calendar month |
| `seasonality` | 24 | Seasonal lag used for MASE denominator |
| `verbose` | `False` | Print progress to stdout |

**Return value** — a `dict` with keys:

| Key | Type | Description |
|-----|------|-------------|
| `metrics` | dict | Scalar scores: `mae`, `rmse`, `wape`, `mape`, `mase` |
| `mae_horizon` | `np.ndarray` (H,) | MAE for each forecast step h=1..H |
| `mae_hour` | `pd.Series` (24,) | MAE grouped by hour of day |
| `predictions` | `pd.DataFrame` | Columns `y_true`, `y_pred` with DatetimeIndex |
| `windows` | list of dicts | Metadata per evaluation window |


---
## 2. Synthetic Dataset

We generate **two full years of synthetic hourly electricity consumption** with
realistic daily and weekly seasonality plus Gaussian noise.


In [ ]:
def make_synthetic_dataset(n_hours: int = 17520, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic hourly energy-consumption data.

    Consumption pattern:
      - Base load ~200 MWh
      - Daily cycle (peak at 18h)
      - Weekly cycle (lower on weekends)
      - Slight annual seasonality
      - Gaussian noise
    """
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2022-01-01", periods=n_hours, freq="h")

    hours   = np.arange(n_hours)
    h_of_d  = idx.hour.values          # 0..23
    d_of_w  = idx.dayofweek.values     # 0=Mon..6=Sun
    d_of_y  = idx.dayofyear.values

    # components
    base    = 200.0
    daily   = 50 * np.sin(2 * np.pi * (h_of_d - 6) / 24)   # peak ~18h
    weekly  = -20 * (d_of_w >= 5).astype(float)             # weekend dip
    annual  = 30 * np.cos(2 * np.pi * d_of_y / 365)         # summer/winter
    noise   = rng.normal(0, 8, size=n_hours)

    # near-zero hours: 02h–04h on weekends (for MAPE demo)
    near_zero_mask = (h_of_d >= 2) & (h_of_d <= 4) & (d_of_w >= 5)
    y = base + daily + weekly + annual + noise
    y[near_zero_mask] = rng.uniform(0.5, 3.0, size=near_zero_mask.sum())
    y = np.clip(y, 0, None)

    return pd.DataFrame({"y": y}, index=idx)


df = make_synthetic_dataset()
print(f"Dataset: {len(df):,} hourly observations  ({df.index[0].date()} → {df.index[-1].date()})")
df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df.index, df["y"], linewidth=0.4, color="steelblue")
axes[0].set_title("Full series (hourly)")
axes[0].set_ylabel("Consumption (MWh)")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# one week
week = df["2022-06-06":"2022-06-12"]
axes[1].plot(week.index, week["y"], marker=".", linewidth=1, color="darkorange")
axes[1].set_title("One week zoom (2022-Jun)")
axes[1].set_ylabel("Consumption (MWh)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))

plt.tight_layout()
plt.show()


---
## 3. Seasonal Naïve Baseline

The **Seasonal Naïve** model forecasts each future step $h$ by copying the value
observed exactly one season ago:

$$\hat{y}_{t+h} = y_{t+h - s}$$

where $s = 24 \times 7 = 168$ (one week in hours).  It is the canonical baseline
for energy forecasting because electricity consumption is strongly weekly-seasonal.


In [ ]:
class SeasonalNaive:
    """Seasonal Naïve forecaster.

    Predicts the next `horizon` steps by repeating the last observed
    seasonal cycle (of length `seasonality`).

    Implements the sklearn-style fit / predict interface expected by
    `backtest()`.
    """

    def __init__(self, seasonality: int = 168, horizon: int = 24):
        """Parameters
        ----------
        seasonality : int
            Number of lags that constitute one full season.
            168 = 24 h × 7 days (weekly pattern).
        horizon : int
            Number of future steps to forecast.
        """
        self.seasonality = seasonality
        self.horizon = horizon
        self._last_season: np.ndarray | None = None

    # ------------------------------------------------------------------
    def fit(self, X, y: np.ndarray):
        """Memorise the last `seasonality` observations of the training series."""
        y = np.asarray(y).ravel()
        self._last_season = y[-self.seasonality:].copy()
        return self

    # ------------------------------------------------------------------
    def predict(self, X=None) -> np.ndarray:
        """Return a flat array of length `horizon` with the seasonal forecast."""
        if self._last_season is None:
            raise RuntimeError("Call fit() before predict().")
        # Tile the last season until we have at least `horizon` values
        n_tiles = int(np.ceil(self.horizon / self.seasonality))
        repeated = np.tile(self._last_season, n_tiles)
        return repeated[: self.horizon]


# Quick sanity check
sn = SeasonalNaive(seasonality=168, horizon=24)
sn.fit(None, y=np.arange(200, dtype=float))
pred = sn.predict()
print("Horizon-24 forecast (first 8 steps):", pred[:8])
print("Expected (last season, first 8 steps):",
      np.arange(200, dtype=float)[-168:][:8])


---
## 4. Error Metrics

We implement all five metrics from scratch so the formulas are transparent.


In [ ]:
# ── Metric implementations ────────────────────────────────────────────────────

def _mae(y_true, y_pred):
    """Mean Absolute Error — average magnitude of error, same unit as y."""
    return float(np.mean(np.abs(y_true - y_pred)))


def _rmse(y_true, y_pred):
    """Root Mean Squared Error — penalises large errors more than MAE."""
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def _wape(y_true, y_pred):
    """Weighted Absolute Percentage Error.

    WAPE = sum|y - y_hat| / sum|y|

    Equivalent to MAE / mean(y) when y >= 0.  Robust to near-zero values
    because it divides by the *total* consumption, not each individual value.
    """
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred)) / denom)


def _mape(y_true, y_pred, eps=1e-8):
    """Mean Absolute Percentage Error.

    MAPE = mean( |y - y_hat| / |y| )

    WARNING: explodes when any y ≈ 0 (division by a tiny number).
    """
    return float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))))


def _mase(y_true, y_pred, y_train, seasonality=24):
    """Mean Absolute Scaled Error.

    MASE = MAE(model) / MAE(seasonal_naive_on_train)

    A score < 1 means the model beats the seasonal naïve on the training set.
    Scale-free — comparable across different datasets and units.
    """
    naive_errors = np.abs(y_train[seasonality:] - y_train[:-seasonality])
    scale = np.mean(naive_errors)
    if scale == 0:
        return float("nan")
    return float(np.mean(np.abs(y_true - y_pred)) / scale)


print("Metrics defined: MAE, RMSE, WAPE, MAPE, MASE")


In [ ]:
# ── Demonstrate on a small example ───────────────────────────────────────────
y_ex   = np.array([100., 150., 200., 250., 300.])
yh_ex  = np.array([ 90., 160., 195., 270., 280.])
train_ex = np.arange(1, 201, dtype=float)   # dummy training series

print(f"MAE  : {_mae(y_ex, yh_ex):.2f} MWh")
print(f"RMSE : {_rmse(y_ex, yh_ex):.2f} MWh")
print(f"WAPE : {_wape(y_ex, yh_ex)*100:.2f} %")
print(f"MAPE : {_mape(y_ex, yh_ex)*100:.2f} %")
print(f"MASE : {_mase(y_ex, yh_ex, train_ex, seasonality=24):.4f}")


### Metric comparison table

| Metric | Unit | Scale-free | Robust to 0 | Penalises large errors |
|--------|------|-----------|------------|------------------------|
| MAE  | same as y | ✗ | ✓ | moderate |
| RMSE | same as y | ✗ | ✓ | **high** |
| WAPE | % | ✓ | ✓ | moderate |
| MAPE | % | ✓ | **✗** | moderate |
| MASE | dimensionless | ✓ | ✓ | moderate |


---
## 5. Running the Backtest with SeasonalNaive


In [ ]:
model = SeasonalNaive(seasonality=168, horizon=24)

config = {
    "horizon":         24,
    "initial_train":   8760,   # first year as training
    "step":            24,     # advance one day at a time
    "monthly_retrain": True,
    "seasonality":     168,
    "verbose":         False,
}

results = backtest(model, df, config)
print("Backtest complete.")
print(f"Evaluation windows : {len(results['windows'])}")
print(f"Predictions shape  : {results['predictions'].shape}")


In [ ]:
# ── Print all scalar metrics ──────────────────────────────────────────────────
print("\n=== Seasonal Naïve — Backtest Metrics ===")
m = results["metrics"]
print(f"  MAE  : {m['mae']:.2f} MWh")
print(f"  RMSE : {m['rmse']:.2f} MWh")
print(f"  WAPE : {m['wape']*100:.2f} %")
print(f"  MAPE : {m['mape']*100:.2f} %")
print(f"  MASE : {m['mase']:.4f}")


---
## 6. MAE by Forecast Horizon (h = 1 … 24)

We expect the error to grow as the horizon increases — the further ahead we
forecast, the less reliable the prediction.


In [ ]:
mae_h = results["mae_horizon"]   # shape (24,)
horizons = np.arange(1, len(mae_h) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(horizons, mae_h, color="steelblue", edgecolor="white", linewidth=0.4)
ax.axhline(mae_h.mean(), color="red", linestyle="--", label=f"Overall MAE = {mae_h.mean():.2f}")
ax.set_xlabel("Forecast horizon h (hours ahead)")
ax.set_ylabel("MAE (MWh)")
ax.set_title("MAE by Forecast Horizon — Seasonal Naïve")
ax.set_xticks(horizons)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Best  horizon (lowest MAE): h = {horizons[mae_h.argmin()]:2d}  ({mae_h.min():.2f} MWh)")
print(f"Worst horizon (highest MAE): h = {horizons[mae_h.argmax()]:2d}  ({mae_h.max():.2f} MWh)")


---
## 7. MAE by Hour of Day

Some hours are inherently harder to forecast (e.g., ramp-up hours 06–09 and
evening peak 17–20) because consumption changes rapidly.


In [ ]:
mae_hour = results["mae_hour"]   # pd.Series indexed 0..23

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mae_hour.index, mae_hour.values, color="darkorange", edgecolor="white")
ax.axhline(mae_hour.mean(), color="navy", linestyle="--",
           label=f"Mean = {mae_hour.mean():.2f} MWh")
ax.set_xlabel("Hour of day (0 = midnight)")
ax.set_ylabel("MAE (MWh)")
ax.set_title("MAE by Hour of Day — Seasonal Naïve")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
worst_hours = mae_hour.nlargest(3)
best_hours  = mae_hour.nsmallest(3)
print("Top-3 hardest hours to forecast (highest MAE):")
print(worst_hours.to_string())
print("\nTop-3 easiest hours to forecast (lowest MAE):")
print(best_hours.to_string())


---
## 8. Christmas Week Error Analysis (Dec 22–28)

Holiday periods break the weekly pattern that the Seasonal Naïve relies on,
so we expect the error to spike during Christmas week.


In [ ]:
preds = results["predictions"]   # DatetimeIndex, columns y_true / y_pred

# Extract Christmas week (2022-12-22 to 2022-12-28)
xmas = preds.loc["2022-12-22":"2022-12-28"].copy()
xmas["error"] = (xmas["y_true"] - xmas["y_pred"]).abs()

# Also pick a 'normal' reference week for comparison
ref = preds.loc["2022-11-07":"2022-11-13"].copy()
ref["error"] = (ref["y_true"] - ref["y_pred"]).abs()

print(f"Christmas week  — MAE: {xmas['error'].mean():.2f} MWh")
print(f"Reference week  — MAE: {ref['error'].mean():.2f} MWh")
print(f"Error inflation : {xmas['error'].mean() / ref['error'].mean():.2f}x")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

for ax, week_df, week_label, color in [
    (axes[0], xmas, "Christmas week (Dec 22–28)", "crimson"),
    (axes[1], ref,  "Reference week  (Nov 07–13)", "steelblue"),
]:
    ax.plot(week_df.index, week_df["y_true"], label="Actual",
            linewidth=1.8, color="black")
    ax.plot(week_df.index, week_df["y_pred"], label="Seasonal Naïve",
            linestyle="--", linewidth=1.5, color=color)
    ax.fill_between(week_df.index, week_df["y_true"], week_df["y_pred"],
                    alpha=0.15, color=color, label="Error")
    ax.set_title(f"{week_label}  (MAE = {week_df['error'].mean():.2f} MWh)")
    ax.set_ylabel("Consumption (MWh)")
    ax.legend(loc="upper right", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))

plt.tight_layout()
plt.show()


In [ ]:
# Daily MAE breakdown across Christmas week
daily_xmas = xmas["error"].resample("D").mean()
daily_xmas.index = daily_xmas.index.strftime("%a %d %b")

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.bar(daily_xmas.index, daily_xmas.values,
              color=["crimson" if v == daily_xmas.max() else "salmon"
                     for v in daily_xmas.values])
ax.axhline(ref["error"].mean(), color="steelblue", linestyle=":",
           label="Reference week MAE")
ax.set_title("Daily MAE — Christmas Week (Dec 22–28)")
ax.set_ylabel("MAE (MWh)")
ax.legend()
plt.tight_layout()
plt.show()


**Observation:** Christmas Day (Dec 25) and Boxing Day (Dec 26) produce the
highest errors because the actual consumption is far below the previous week's
values that the Seasonal Naïve copies.  This motivates adding a **holiday
calendar** as a feature in more sophisticated models.


---
## 9. Walk-Forward Validation with Monthly Retraining

In real deployments, models are periodically retrained on the latest data.
The `backtest()` function supports this via `monthly_retrain=True`.

Below we compare:
- **No retraining** — model fitted once on the initial training set.
- **Monthly retraining** — model refitted every time a new month starts.


In [ ]:
base_config = {
    "horizon":       24,
    "initial_train": 8760,
    "step":          24,
    "seasonality":   168,
    "verbose":       False,
}

# Run both variants
results_no_retrain = backtest(
    SeasonalNaive(seasonality=168, horizon=24),
    df,
    {**base_config, "monthly_retrain": False},
)

results_monthly = backtest(
    SeasonalNaive(seasonality=168, horizon=24),
    df,
    {**base_config, "monthly_retrain": True},
)

print("=" * 50)
print(f"{'Metric':<8} {'No retraining':>15} {'Monthly retrain':>17}")
print("-" * 50)
for k in ["mae", "rmse", "wape", "mase"]:
    v_no  = results_no_retrain["metrics"][k]
    v_mo  = results_monthly["metrics"][k]
    unit  = " %" if k in ("wape",) else ""
    scale = 100  if k in ("wape",) else 1
    print(f"{k.upper():<8} {v_no*scale:>15.3f}{unit}  {v_mo*scale:>15.3f}{unit}")
print("=" * 50)


In [ ]:
# Visualise cumulative MAE over the evaluation period
pred_no  = results_no_retrain["predictions"].copy()
pred_mo  = results_monthly["predictions"].copy()

pred_no["abs_err"] = (pred_no["y_true"] - pred_no["y_pred"]).abs()
pred_mo["abs_err"] = (pred_mo["y_true"] - pred_mo["y_pred"]).abs()

daily_no = pred_no["abs_err"].resample("D").mean()
daily_mo = pred_mo["abs_err"].resample("D").mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_no.index, daily_no.rolling(7).mean(),
        label="No retraining (7-day rolling MAE)", color="tomato")
ax.plot(daily_mo.index, daily_mo.rolling(7).mean(),
        label="Monthly retraining (7-day rolling MAE)", color="steelblue")
ax.set_ylabel("MAE (MWh)")
ax.set_title("Walk-Forward MAE: No Retraining vs Monthly Retraining")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### How many fit() calls did monthly retraining trigger?


In [ ]:
# Inspect the windows metadata to count retrain events
windows_df = pd.DataFrame(results_monthly["windows"])
windows_df["month"] = pd.to_datetime(windows_df["test_start"]).dt.to_period("M")
retrain_months = windows_df.drop_duplicates(subset="month")["month"]

print(f"Total evaluation windows  : {len(windows_df)}")
print(f"Number of model refits    : {len(retrain_months)}")
print("Months retrained:", retrain_months.tolist())


---
## 10. Why MAPE Fails with Near-Zero Consumption

MAPE divides the absolute error by the **actual value**:

$$\text{MAPE} = \frac{1}{n} \sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|$$

When $y_t \approx 0$ (e.g., a building at 3 AM on a Sunday), even a tiny absolute
error produces a **huge percentage error**, skewing the overall metric.


In [ ]:
# Illustrative example: identical absolute error, wildly different MAPE
print("Scenario A — normal consumption")
y_a   = np.array([200.0])
yh_a  = np.array([210.0])   # 10 MWh error
print(f"  Actual: {y_a[0]:.0f} MWh  |  Predicted: {yh_a[0]:.0f} MWh")
print(f"  Absolute error : {abs(y_a[0]-yh_a[0]):.1f} MWh")
print(f"  MAPE           : {_mape(y_a, yh_a)*100:.1f} %")

print()
print("Scenario B — near-zero consumption (same absolute error!)")
y_b   = np.array([1.0])
yh_b  = np.array([11.0])    # 10 MWh error — same as above
print(f"  Actual: {y_b[0]:.1f} MWh  |  Predicted: {yh_b[0]:.1f} MWh")
print(f"  Absolute error : {abs(y_b[0]-yh_b[0]):.1f} MWh")
print(f"  MAPE           : {_mape(y_b, yh_b)*100:.0f} %  ← explosion!")


In [ ]:
# Show the effect on the real dataset
# Identify near-zero hours in the evaluation period
preds_all = results_monthly["predictions"].copy()
preds_all["near_zero"] = preds_all["y_true"] < 5.0
preds_all["mape_t"] = (preds_all["y_true"] - preds_all["y_pred"]).abs() / \
                       (preds_all["y_true"].abs() + 1e-8)

normal   = preds_all[~preds_all["near_zero"]]
nearzero = preds_all[ preds_all["near_zero"]]

print(f"Normal hours     ({len(normal):5,} pts) — mean MAPE: {normal['mape_t'].mean()*100:6.1f} %")
print(f"Near-zero hours  ({len(nearzero):5,} pts) — mean MAPE: {nearzero['mape_t'].mean()*100:6.1f} %")
print()
print("Overall MAPE (contaminated by near-zero):",
      f"{_mape(preds_all['y_true'].values, preds_all['y_pred'].values)*100:.1f} %")
print("WAPE (robust):      ",
      f"{_wape(preds_all['y_true'].values, preds_all['y_pred'].values)*100:.1f} %")


In [ ]:
# Visualise the MAPE blow-up
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: point-wise MAPE distribution (capped at 500% for readability)
mape_capped = preds_all["mape_t"].clip(upper=5) * 100
axes[0].hist(mape_capped[~preds_all["near_zero"]], bins=80, alpha=0.7,
             color="steelblue", label="Normal hours", density=True)
axes[0].hist(mape_capped[preds_all["near_zero"]], bins=80, alpha=0.7,
             color="crimson", label="Near-zero hours", density=True)
axes[0].set_xlabel("Point-wise MAPE (%, capped at 500)")
axes[0].set_ylabel("Density")
axes[0].set_title("MAPE distribution")
axes[0].legend()

# Right: bar comparing MAPE vs WAPE
metrics_compare = {
    "MAPE\n(all)": _mape(preds_all['y_true'].values,
                         preds_all['y_pred'].values) * 100,
    "MAPE\n(normal only)": _mape(normal['y_true'].values,
                                  normal['y_pred'].values) * 100,
    "WAPE\n(robust)": _wape(preds_all['y_true'].values,
                             preds_all['y_pred'].values) * 100,
}
colors = ["crimson", "darkorange", "seagreen"]
axes[1].bar(list(metrics_compare.keys()),
            list(metrics_compare.values()), color=colors)
axes[1].set_ylabel("%")
axes[1].set_title("MAPE vs WAPE comparison")
for i, (k, v) in enumerate(metrics_compare.items()):
    axes[1].text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=10)

plt.tight_layout()
plt.show()


### Conclusion on MAPE

| Situation | Recommended metric |
|-----------|--------------------|
| Consumption always > 0, no near-zero | MAPE is acceptable |
| Some hours near zero (nights, holidays) | **WAPE** or **MASE** |
| Need scale-free comparison across regions/meters | **MASE** |
| Need absolute magnitude for operations | **MAE** or **RMSE** |

> **Rule of thumb for energy forecasting:** always report WAPE alongside MAE.
> Avoid MAPE when any actual consumption value is close to zero.


---
## 11. Summary

| Component | Key takeaway |
|-----------|-------------|
| `backtest()` | Walk-forward evaluation; use `config` to tune horizon, retraining schedule, and seasonality |
| Seasonal Naïve | Strong, fast baseline: copies values from 7 days ago |
| MAE by horizon | Error grows with h; helps diagnose model degradation at longer leads |
| MAE by hour | Ramp-up / peak hours are hardest; guides feature engineering |
| Christmas week | Holiday anomalies inflate error ×2–×4; model should see a holiday calendar |
| Monthly retraining | Re-fitting on fresh data typically reduces error; confirm with held-out test |
| MAPE failure | Near-zero actuals inflate MAPE artificially; prefer WAPE or MASE |


In [ ]:
print("Notebook completed successfully.")
